# EPI Recorder v4.4.6 — Meridian Bank Loan Desk Demo

**A loan bot approves ₹75,000 solo — no risk check, no human signoff. EPI keeps the receipt.**

In 6 cells you will see a real banking-governance story:

1. The bot **breaks three rules at once** (approve without risk assessment, above-limit amount without human approval) → the violation is **preserved as evidence**.
2. The fixed bot runs the full desk flow (KYC → risk assessment → human approval → decision) → **strict verification PASSES**, org-pinned to Meridian.
3. A forged copy **fails**. The evidence viewer opens with real data.

*Runtime ~3 minutes. No API key needed — the credit model is simulated.*

**Click Runtime → Run All.**

In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
!pip install -q epi-recorder
import epi_recorder, epi_core._version as _v
print("epi-recorder", _v.get_version())
!epi version

In [ ]:
# @title 2. Bank identity + lending policy { display-mode: "form" }
# Meridian Bank as an org: a root key (board-held, offline), a daily sealing
# key for the loan-desk bot, and the lending policy book.
!epi keys generate --name org-root
!epi keys generate --name loan-desk-bot
!epi policy init --profile finance.loan-underwriting --yes
!epi policy lint

import os
from epi_core.keys import KeyManager
from epi_core.org_bundle import fingerprint_pubkey

km = KeyManager()  # EPI_HOME-aware: newcomer keys land where you look
root_pub = km._load_public_key_raw_bytes("org-root").hex()
seal_pub = km._load_public_key_raw_bytes("loan-desk-bot").hex()
ORG_ROOT = fingerprint_pubkey(root_pub)
os.environ["EPI_ORG_ROOT"] = ORG_ROOT  # bound into every manifest sealed below

# not_before is backdated so this bundle covers loans booked from Jan 2025 on.
# (A bundle issued "now" cannot vouch for earlier seals — try omitting it later.)
!epi org bundle issue --bundle-id meridian-bank --root-key org-root --key loan-desk-bot={seal_pub} --not-before 2025-01-01T00:00:00Z --out org-bundle.json
print("ORG_ROOT:", ORG_ROOT)

In [ ]:
# @title 3. The violation — Rs.75,000 approved solo, no checks { display-mode: "form" }
%%writefile loan_bad.py
import os
from epi_recorder import record

ORG_ROOT = os.environ["EPI_ORG_ROOT"]

applicant = {"business": "Sharma Electronics", "credit_score": 680,
               "annual_revenue": 850000, "requested": 75000}
print(f"Applicant: {applicant['business']} asks Rs.{applicant['requested']:,}")

with record("bad.epi", workflow_name="Loan Underwriting",
               default_key_name="loan-desk-bot", org_root=ORG_ROOT) as epi:
    # No KYC. No risk assessment. No human approval for 75,000 (> 10,000).
    # Just the approval — everything the policy book forbids.
    epi.log_step("agent.decision", {
        "action": "approve_loan",
        "amount": applicant["requested"],
        "applicant": applicant["business"],
        "risk_assessment": None,
        "human_approval": None,
    })
    print("Decision recorded: APPROVED Rs.75,000, checks=None")

!python loan_bad.py
print("\n--- strict verification (seal is intact, signer unknown) ---")
!epi verify bad.epi --policy strict
print("\n--- audit: the violation is preserved as evidence ---")
!epi audit bad.epi --format json > audit_bad.json

import json
rep = json.load(open("audit_bad.json"))
fa = rep["pipeline"]["fault_analysis"]
print("fault_detected:", fa["fault_detected"], "| verdict:", fa["verdict"])
print("compliance:", rep["compliance_score"]["percentage"], "% ->", rep["compliance_score"]["rating"])

In [ ]:
# @title 4. The compliant run — full desk flow, identity pinned { display-mode: "form" }
%%writefile loan_good.py
import os
from epi_recorder import record

ORG_ROOT = os.environ["EPI_ORG_ROOT"]

with record("good.epi", workflow_name="Loan Underwriting",
               default_key_name="loan-desk-bot", org_root=ORG_ROOT) as epi:
    epi.log_step("llm.request", {"model": "underwriting-llm",
                                    "prompt": "review application SME-2210"})
    epi.log_step("llm.response", {"assessment": "credit 680, revenue 850k, GST compliant"})
    epi.log_step("tool.call", {"tool": "verify_identity", "call_id": "v1"})
    epi.log_step("tool.response", {"tool": "verify_identity", "call_id": "v1", "kyc": "pass"})
    epi.log_step("tool.call", {"tool": "risk_assessment", "call_id": "r1"})
    epi.log_step("tool.response", {"tool": "risk_assessment", "call_id": "r1",
                                      "credit_score": 680, "credit_limit": 200000,
                                      "risk": "medium"})
    epi.log_step("agent.approval.request", {"action": "human_approval",
                                               "reason": "75000 above 10000 threshold"})
    epi.log_step("agent.approval.response", {"action": "human_approval", "approved": True,
                                                "approved_by": "meera.k@meridian.example"})
    epi.log_step("agent.decision", {"action": "approve_loan", "amount": 75000,
                                      "plan": "approve within 200000 limit, quarterly review"})
    print("Decision recorded: APPROVED Rs.75,000, KYC pass, risk medium, signed meera.k")

!python loan_good.py
print("\n--- first look: valid seal, UNKNOWN signer (anyone can mint a key) ---")
!epi verify good.epi
print("\n--- pin the signer, then strict ---")
!epi keys trust good.epi --name loan-desk-bot
!epi verify good.epi --policy strict
print("\n--- org binding: did this come from Meridian? ---")
!epi org bundle verify good.epi org-bundle.json
print("\n--- full audit ---")
!epi audit good.epi

In [ ]:
# @title 5. Tamper test — flip one byte, forgery must fail { display-mode: "form" }
from pathlib import Path

original = Path("good.epi")
data = bytearray(original.read_bytes())
data[len(data) // 2] ^= 0xFF  # one bit-flip deep inside the evidence
Path("FORGED.epi").write_bytes(bytes(data))
print(f"Flipped 1 byte at offset {len(data)//2} of {len(data)}")
print("\n" + "=" * 55 + "\nORIGINAL:")
!epi verify good.epi --policy strict > /dev/null && echo "original: PASS (exit 0)"
print("\n" + "=" * 55 + "\nFORGED (1 byte changed):")
!epi verify FORGED.epi; echo "forged exit code: $?"
print("=" * 55)
Path("FORGED.epi").unlink()

In [ ]:
# @title 6. Evidence viewer — open the sealed loan file { display-mode: "form" }
# export-html builds a FRESH standalone viewer with current client crypto
# (no pre-verified overrides — the badge verifies for real in your browser).
!epi export-html good.epi --output loan_evidence.html

from pathlib import Path
from IPython.display import display, HTML

html_text = Path("loan_evidence.html").read_text(encoding="utf-8")
display(HTML(
    f'<div style="border:2px solid #2563eb;border-radius:8px;overflow:hidden;margin:10px 0">'
    f'<div style="background:#2563eb;color:white;padding:10px 16px;font-weight:bold">'
    f"EPI Evidence — good.epi (sealed, org-pinned, strict PASS)</div>"
    f'<iframe srcdoc="{html_text.replace(chr(34), chr(39))}" '
    f'width="100%" height="600" style="border:none"></iframe></div>'
))

try:
    from google.colab import files
    files.download("good.epi")
    files.download("loan_evidence.html")
    files.download("org-bundle.json")
    print("Downloaded: good.epi + loan_evidence.html + org-bundle.json")
    print("Send all three to an auditor: epi org bundle verify good.epi org-bundle.json")
except Exception:
    pass